In [3]:
import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ============================================================
# CONFIGURATION & SEEDING
# ============================================================


DATA_ROOT = "DeFungi Dataset"
BASE_OUTPUT_DIR = "OD-Abs/Cross Validation Final Results Beta-VAE"
ANOMALY_DIR = "DiBaSandDeepBacs"


# Hyperparameters
IMG_SIZE = 256
BATCH_SIZE = 32
LATENT_DIM = 32
LR = 1e-5
N_FOLDS = 5
EPOCHS_PER_FOLD = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random Seed set to: {seed}")

seed_everything(42)
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

Random Seed set to: 42


In [4]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

# ============================================================
# DATASET PREPARATION
# ============================================================

class FungiDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.files = file_paths
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        try:
            # Convert to Grayscale
            img = Image.open(img_path).convert("L")
            if self.transform:
                img = self.transform(img)
            return img
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros((1, IMG_SIZE, IMG_SIZE))

all_image_paths = []
print("Scanning dataset:")


valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
if os.path.exists(DATA_ROOT):
    for f in os.listdir(DATA_ROOT):
        file_path = os.path.join(DATA_ROOT, f)

        if os.path.isfile(file_path) and f.lower().endswith(valid_exts):
            all_image_paths.append(file_path)
else:
    print(f"Error: The directory '{DATA_ROOT}' does not exist")

print(f"Found {len(all_image_paths)} total images")

# Define Transform
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

Scanning dataset:
Found 9114 total images


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class ResNetVAE(nn.Module):
    def __init__(self, latent_dim=32, img_size=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.img_size = img_size

        # ------------------------
        # ENCODER (Pretrained ResNet18)
        # ------------------------
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Modify First Layer for Grayscale
        original_conv1 = resnet.conv1
        self.enc_conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        with torch.no_grad():
            self.enc_conv1.weight.copy_(torch.sum(original_conv1.weight, dim=1, keepdim=True))

        # Feature Extraction
        # (256 -> 128 -> 64 -> 32 -> 16 -> 8)
        self.encoder_backbone = nn.Sequential(
            self.enc_conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1, # 64 channels, 64x64
            resnet.layer2, # 128 channels, 32x32
            resnet.layer3, # 256 channels, 16x16
            resnet.layer4  # 512 channels, 8x8
        )

        # Latent Projections
        self.flatten_dim = 512 * 8 * 8

        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.flatten_dim)

        # ------------------------
        # DECODER
        # ------------------------

        self.decoder = nn.Sequential(
            # Block 1: 8x8 -> 16x16
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(512, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 2: 16x16 -> 32x32
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 3: 32x32 -> 64x64
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(128, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 4: 64x64 -> 128x128
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(64, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 5: 128x128 -> 256x256
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            # Final Output Layer
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        batch_size = x.size(0)

        # 1. Encode
        features = self.encoder_backbone(x)
        features_flat = torch.flatten(features, 1)

        mu = self.fc_mu(features_flat)
        logvar = self.fc_logvar(features_flat)

        logvar = torch.clamp(logvar, -10, 10)

        # 2. Sample
        z = self.reparameterize(mu, logvar)

        # 3. Decode
        dec_in = self.fc_dec(z)

        # Update View Shape to match 8x8 feature map
        dec_in = dec_in.view(batch_size, 512, 8, 8)

        recon = self.decoder(dec_in)

        return recon, mu, logvar

In [6]:
# Loss Function
def vae_loss_with_annealing(recon, x, mu, logvar, beta=1):
    recon_loss = F.l1_loss(recon, x, reduction='none')
    recon_loss = recon_loss.view(recon_loss.size(0), -1).sum(dim=1)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    total_loss_per_sample = recon_loss + (beta * kl_loss)
    return total_loss_per_sample.mean(), recon_loss, kl_loss

# Evaluation Loss Helper
def evaluation_loss_function(recon, x, mu, logvar):
    recon_loss = F.l1_loss(recon, x, reduction='none').view(recon.size(0), -1).sum(dim=1)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return recon_loss + kl_loss


In [7]:
# ============================================================
# EVALUATION LOGIC
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import os
from torch.utils.data import DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim


def compute_metrics(orig_np, recon_np):
    criterion = nn.L1Loss()

    if orig_np.shape != recon_np.shape:
        raise ValueError(f"Shape mismatch: {orig_np.shape} vs {recon_np.shape}")

    orig_01 = np.clip((orig_np + 1) / 2, 0, 1)
    recon_01 = np.clip((recon_np + 1) / 2, 0, 1)

    orig_t = torch.from_numpy(orig_np).unsqueeze(0).unsqueeze(0).float()
    recon_t = torch.from_numpy(recon_np).unsqueeze(0).unsqueeze(0).float()

    l1_loss_val = criterion(recon_t, orig_t).item()
    mse_val = torch.mean((orig_t - recon_t) ** 2).item()

    psnr_val = psnr(orig_01, recon_01, data_range=1.0)
    ssim_val = ssim(orig_01, recon_01, data_range=1.0)

    signal_power = np.mean(orig_np ** 2)
    noise_power = mse_val + 1e-12
    snr_val = 10 * np.log10(signal_power / noise_power)

    return {
        "L1_Loss": l1_loss_val,
        "MSE": mse_val,
        "PSNR": psnr_val,
        "SSIM": ssim_val,
        "SNR": snr_val
    }


def evaluate_subset(model, file_paths, species_name="fungi"):
    """Evaluates a list of file paths."""
    results = []
    model.eval()

    # Create temp dataset/loader for this subset
    eval_ds = FungiDataset(file_paths, transform=transform)
    # Batch size 1 for accurate metrics per image
    eval_loader = DataLoader(eval_ds, batch_size=1, shuffle=False)

    print(f"Evaluating {len(file_paths)} images for {species_name}")

    with torch.no_grad():
        for i, img in enumerate(eval_loader):
            img = img.to(DEVICE)
            fname = os.path.basename(file_paths[i])

            recon, mu, logvar = model(img)
            recon = torch.clamp(recon, -1, 1)

            # Anomaly Score
            loss_t = evaluation_loss_function(recon, img, mu, logvar)
            model_loss = loss_t.item()

            # Visual Metrics
            recon_np = recon.squeeze().cpu().numpy()
            orig_np = img.squeeze().cpu().numpy()

            metrics = compute_metrics(orig_np, recon_np)
            metrics.update({
                "filename": fname,
                "species": species_name,
                "is_fungi": 1 if species_name == "fungi" else 0,
                "model_loss": model_loss
            })
            results.append(metrics)

    return results

def evaluate_external_folder(model, folder_path, species_name):
    """Legacy wrapper to evaluate external anomaly folders (like bacteria)"""
    if not os.path.exists(folder_path): return []
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
             if f.lower().endswith(('.png', '.jpg', '.tif', '.tiff', '.jpeg'))]
    return evaluate_subset(model, files, species_name)

In [ ]:
# ============================================================
# CROSS VALIDATION LOOP
# ============================================================

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import precision_recall_curve


kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X = np.array(all_image_paths)
print(f"Starting {N_FOLDS}-Fold Cross Validation:")


for fold, (train_val_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n==================== FOLD {fold+1}/{N_FOLDS} ====================")

    # 1. Prepare Directory
    fold_dir = os.path.join(BASE_OUTPUT_DIR, f"fold_{fold+1}")
    os.makedirs(fold_dir, exist_ok=True)

    # 2. Data Splitting
    # X_test is 20% of total data (determined by K-Fold)
    X_test = X[test_idx]

    # X_train_val is 80% of total data
    X_train_val = X[train_val_idx]

    # Split the 80% into Train (60% total) and Val (20% total)
    X_train, X_val = train_test_split(X_train_val, test_size=0.25, random_state=42)

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    # 3. Create Datasets & Loaders
    ds_train = FungiDataset(X_train, transform=transform)
    ds_val = FungiDataset(X_val, transform=transform)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # 4. Initialize Fresh Model
    model = ResNetVAE(latent_dim=LATENT_DIM, img_size=IMG_SIZE).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # 5. Training Loop
    best_val_loss = float('inf')
    best_model_path = os.path.join(fold_dir, "best_model.pth")

    for epoch in range(EPOCHS_PER_FOLD):
        model.train()
        total_loss = 0
        beta = 3

        for imgs in loader_train:
            imgs = imgs.to(DEVICE)
            recon, mu, logvar = model(imgs)
            loss_val, r_loss, k_loss = vae_loss_with_annealing(recon, imgs, mu, logvar, beta=beta)

            optimizer.zero_grad()
            loss_val.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss_val.item() * imgs.size(0)

        avg_train = total_loss / len(ds_train)

        # Validation
        model.eval()
        total_val = 0
        with torch.no_grad():
            for imgs in loader_val:
                imgs = imgs.to(DEVICE)
                recon, mu, logvar = model(imgs)
                l, _, _ = vae_loss_with_annealing(recon, imgs, mu, logvar, beta=beta)
                total_val += l.item() * imgs.size(0)
        avg_val = total_val / len(ds_val)

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train {avg_train:.4f} | Val {avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), best_model_path)

    # 6. Load Best Model for Evaluation
    print(f"Training Done. Loading best model for evaluation...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()

    # =========================================================================
    # DUAL THRESHOLD CALCULATION
    # =========================================================================

    # Split Anomalies (50% Tuning / 50% Testing)
    all_anom_files = [os.path.join(ANOMALY_DIR, f) for f in os.listdir(ANOMALY_DIR)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]

    val_anom_files, test_anom_files = train_test_split(all_anom_files, test_size=0.5, random_state=fold)
    print(f"Anomaly Split: {len(val_anom_files)} Tuning | {len(test_anom_files)} Testing")

    # Calculate Threshold 1: Mean + 2*Std
    train_scores = []
    with torch.no_grad():
        for imgs in loader_train:
            imgs = imgs.to(DEVICE)
            recon, mu, logvar = model(imgs)
            l1_loss = F.l1_loss(recon, imgs, reduction='none').view(imgs.size(0), -1).mean(dim=1)
            train_scores.append(l1_loss.cpu().numpy())
    train_scores = np.concatenate(train_scores)

    thresh_mean_std = train_scores.mean() + 2 * train_scores.std()
    print(f"Threshold V1 (Mean+2Std): {thresh_mean_std:.6f}")

    # Calculate Threshold 2: Grid Search
    print("Starting Grid Search for Threshold V2")

    # 1. Get Normal Validation Scores
    val_normal_scores = []
    with torch.no_grad():
        for imgs in loader_val:
            imgs = imgs.to(DEVICE)
            recon, mu, logvar = model(imgs)
            l1_loss = F.l1_loss(recon, imgs, reduction='none').view(imgs.size(0), -1).mean(dim=1)
            val_normal_scores.append(l1_loss.cpu().numpy())
    val_normal_scores = np.concatenate(val_normal_scores)

    # 2. Get Anomaly Validation Scores (Tuning Set)
    ds_anom_val = FungiDataset(val_anom_files, transform=transform)
    loader_anom_val = DataLoader(ds_anom_val, batch_size=BATCH_SIZE, shuffle=False)

    val_anom_scores = []
    with torch.no_grad():
        for imgs in loader_anom_val:
            imgs = imgs.to(DEVICE)
            recon, mu, logvar = model(imgs)
            l1_loss = F.l1_loss(recon, imgs, reduction='none').view(imgs.size(0), -1).mean(dim=1)
            val_anom_scores.append(l1_loss.cpu().numpy())
    val_anom_scores = np.concatenate(val_anom_scores)

    # 3. Grid Search Logic
    y_true_val = np.concatenate([np.zeros(len(val_normal_scores)), np.ones(len(val_anom_scores))])
    y_scores_val = np.concatenate([val_normal_scores, val_anom_scores])

    precisions, recalls, thresholds = precision_recall_curve(y_true_val, y_scores_val)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

    best_idx = np.argmax(f1_scores)
    thresh_grid = thresholds[best_idx]
    print(f"Threshold V2 (Grid Search): {thresh_grid:.6f} (F1: {f1_scores[best_idx]:.4f})")

    # =========================================================================
    # FINAL EVALUATION (Apply both thresholds to Test Set)
    # =========================================================================
    print("Running Final Evaluation on Test Split:")

    # Evaluate Normal Test Set
    results_normal_test = evaluate_subset(model, X_test, species_name="fungi")

    # Evaluate Anomaly Test Set (Test Split)
    results_anom_test = evaluate_subset(model, test_anom_files, species_name="bacteria")

    # Combine Results
    df_fold = pd.DataFrame(results_normal_test + results_anom_test)

    # Save Both Thresholds
    df_fold['Threshold_MeanStd'] = thresh_mean_std
    df_fold['Threshold_Grid'] = thresh_grid

    df_fold['True_Label_Binary'] = df_fold['species'].apply(lambda x: 0 if x == 'fungi' else 1)

    # Generate Two Sets of Predictions
    # Prediction V1: Using Statistical Threshold
    df_fold['Pred_MeanStd'] = df_fold['L1_Loss'].apply(lambda x: 1 if x > thresh_mean_std else 0)

    # Prediction V2: Using Grid Search Threshold
    df_fold['Pred_Grid'] = df_fold['L1_Loss'].apply(lambda x: 1 if x > thresh_grid else 0)

    # Save
    csv_path = os.path.join(fold_dir, "test_results.csv")
    df_fold.to_csv(csv_path, index=False)
    print(f"   ✅ Saved fold results to {csv_path}")

    # Cleanup
    del model
    del optimizer
    torch.cuda.empty_cache()

print("Cross Validation Complete!")